# 07 · Inside the detector: where damage enters YOLOv8n, and what restoration undoes

**Part A of *Restoring for the Machine*.** No training. Hypotheses H1–H6 were written in [`PLAN.md`](../PLAN.md)
before this notebook was run; the last section computes every verdict from the numbers.

**The question.** Notebook 05 showed *that* restoration helps detection for noise and hurts it for CLAHE in low light,
and 05b showed that per-image PSNR says almost nothing about *which* images it helps. Here we look inside the detector:
hook every layer of YOLOv8n, and measure how far the features of a damaged (or restored) image are from the features
of the clean image, layer by layer.

| Section | Hypotheses |
|---|---|
| 6 · Depth profiles: where does damage show up? | H1a, H1b |
| 7 · Does DnCNN restore deep features? BM3D vs DnCNN | H2a–c, H5 |
| 8 · Low light: why CLAHE fails | H3a–d |
| 9 · Does feature distance predict per-image AP loss? | H4a, H4b |
| 10 · Ranking restorers | H6 |
| 11 · Scorecard, and the layers for Part B | all |

**Requirements:** Kaggle, **GPU T4**, **Internet ON**. Run as a commit (Save Version → Save & Run All). About 1.5–2 h;
set `QUICK = True` first for a ~10-minute check on 40 images.

**Inputs** (Add Input):
- the **dcnn-last** dataset (`shourjabanerjee/dcnn-last`) for `dncnn_last.pt`;
- the code: either nothing (the first cell clones `github.com/onigiri4711/restoring-for-the-machine`), or the repo
  uploaded as a Kaggle dataset (found automatically if attached).

**Outputs** (`/kaggle/working/outputs/07`): `runs.csv` (mAP, PSNR per run), `per_image.csv` (AP, PSNR per image and
run), `feature_distances.csv.gz` (every image × layer × run), `scorecard_07.json`, `layers_for_partB.json`, figures.

In [ ]:
!pip -q install ultralytics pycocotools bm3d

In [ ]:
import os, sys, glob, json, time, subprocess

# --- the project code (rftm/ package + prior results from Degrade -> Restore -> Detect) ---------------
REPO_URL = "https://github.com/onigiri4711/restoring-for-the-machine"
attached = sorted(glob.glob("/kaggle/input/**/rftm/__init__.py", recursive=True))
if attached:
    REPO = os.path.dirname(os.path.dirname(attached[0]))
    print("using the attached copy of the repo:", REPO)
else:
    REPO = "/kaggle/working/restoring-for-the-machine"
    if os.path.exists(REPO):                       # a clone from an earlier run in this session: update it
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    print("cloned", REPO_URL, "@", subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                                                    capture_output=True, text=True).stdout.strip())
sys.path.insert(0, REPO); sys.path.insert(0, f"{REPO}/tests")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

from rftm.degradations import DEGRADATIONS, degrade
from rftm import restorers as RS, coco as C, features as F, analysis as A, plots as P
from rftm.kaggle import resolve_checkpoint

QUICK = False                     # True: the first 40 images of the subset, ~10 minutes, to check the pipeline
N_IMAGES = 300
SEED = 0
OUT_DIR = "/kaggle/working/outputs/07" + ("_quick" if QUICK else ""); os.makedirs(OUT_DIR, exist_ok=True)
COCO_DIR = "/kaggle/working/coco"
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "turn on Settings -> Accelerator -> GPU T4"
torch.backends.cudnn.benchmark = False      # feature maps should be repeatable run to run
CKPT_PATH = resolve_checkpoint()
print("device:", device, "| checkpoint:", CKPT_PATH, "| QUICK:", QUICK)

## 1 · The same 300 COCO images as notebooks 05 and 05b

The subset is regenerated with notebook 05's rule and checked against the frozen id list in `prior/05/subset_ids.json`.
`POS[image_id]` is the image's position in the **full** 300-image list: damage seeds use it, so in `QUICK` mode the
first 40 images get exactly the same damage as in the full run (and as in 05b).

In [ ]:
C.download_coco(COCO_DIR)
coco_gt = C.load_coco_gt(COCO_DIR)
all_ids = C.subset_ids(coco_gt, N_IMAGES, SEED)            # asserts it equals notebook 05's subset
POS = {iid: k for k, iid in enumerate(all_ids)}
img_ids = all_ids[:40] if QUICK else all_ids
clean_imgs = {iid: C.load_coco_rgb(coco_gt, COCO_DIR, iid) for iid in img_ids}     # ~1 GB for 300 images
n_obj = sum(len(coco_gt.getAnnIds(imgIds=i, iscrowd=False)) for i in img_ids)
print(f"{len(img_ids)} images, {n_obj} objects")

## 2 · YOLOv8n with forward hooks

`FeatureTap` (in `rftm/features.py`) registers a forward hook on every YOLO layer except Upsample, Concat and Detect,
plus a pre-hook that records the input tensor after letterboxing. One `predict()` call then gives both the detections
(exactly as in notebook 05) and the feature maps. The hooks only read tensors; they do not change the output.

Clean and damaged versions of an image have the same size, so YOLO letterboxes them identically and their maps line up
cell by cell. Distances use only the cells that cover the real image, not the grey padding.

In [ ]:
detector = YOLO("yolov8n.pt")
cat_ids = C.yolo_to_coco_ids(detector, coco_gt)
probe = clean_imgs[img_ids[0]]
_ = C.detect(detector, cat_ids, probe, img_ids[0], device)       # warm-up: builds and fuses the predictor's model
tap = F.FeatureTap(detector)

def detect_and_tap(img, iid):
    # One predict() -> (COCO detections, {layer: feature map}).
    res = C.detect(detector, cat_ids, img, iid, device)
    return res, tap.pop()

res_a, fa = detect_and_tap(probe, img_ids[0])
res_b, fb = detect_and_tap(probe, img_ids[0])
layers_df = pd.DataFrame(F.describe_layers(tap, fa))
print(layers_df.to_string(index=False))
d_same = F.compare_feature_sets(fa, fb, *probe.shape[:2])
worst = max(v["cos"] for v in d_same.values())
print(f"\nsame image twice: largest cosine distance over layers = {worst:.2e} (GPU run-to-run noise floor)")
assert worst < 1e-4, "features are not repeatable: distances would be dominated by GPU noise"
assert len(res_a) == len(res_b) and all(abs(x["score"] - y["score"]) < 1e-5 for x, y in zip(res_a, res_b))
assert set(A.NECK) <= set(map(str, tap.layers)), "the neck output layers 15/18/21 are not hooked"
layers_df.to_csv(f"{OUT_DIR}/layers.csv", index=False)

# The letterbox crop must match what ultralytics did: everything outside the box is grey padding (114/255)
pad_iid = next((i for i in img_ids if clean_imgs[i].shape[0] % 32 or clean_imgs[i].shape[1] % 32), None)
if pad_iid is not None:
    _, fp = detect_and_tap(clean_imgs[pad_iid], pad_iid)
    x = fp["input"][0]; H_in, W_in = x.shape[1:]
    t_, b_, l_, r_ = F.letterbox_valid_box(*clean_imgs[pad_iid].shape[:2], H_in, W_in)
    outside = torch.cat([x[:, :t_].flatten(), x[:, b_:].flatten(), x[:, :, :l_].flatten(), x[:, :, r_:].flatten()])
    print(f"image {pad_iid}: {clean_imgs[pad_iid].shape[:2]} -> input {(H_in, W_in)}, real image rows {t_}-{b_}, "
          f"cols {l_}-{r_}; padding pixels checked: {outside.numel()}")
    assert outside.numel() == 0 or (outside - 114 / 255).abs().max() < 2e-3, "letterbox geometry differs from ultralytics'"
    edge = torch.stack([x[:, t_].mean(), x[:, b_ - 1].mean()])
    assert ((edge - 114 / 255).abs() > 1e-4).all(), "first/last image row looks like padding: crop is off by one"

## 3 · Checks on the distance functions

- identical maps → every distance 0;
- CKA ignores rotations and scaling of the channel space; cosine ignores scale; relative L2 does not;
- the letterbox crop keeps exactly the cells that overlap the real image.

These are the same tests as `tests/test_features.py`.

In [ ]:
import test_features as TF
TF.test_identical_maps_have_zero_distance(); TF.test_cka_invariances_and_scale_sensitivity(); TF.test_letterbox_crop()

# On a real image: noise must move features, and a stronger noise must move them further
rng = np.random.default_rng(0)
d = {s: F.compare_feature_sets(fa, detect_and_tap(np.clip(probe + rng.normal(0, s / 255, probe.shape), 0, 1).astype(np.float32),
                                                   img_ids[0])[1], *probe.shape[:2])[0]["cos"] for s in (10, 50)}
print("layer-0 cosine distance, sigma 10 vs 50:", {k: round(v, 4) for k, v in d.items()})
assert d[50] > d[10] > 1e-3
print("distance checks passed")

## 4 · Clean baseline

Must reproduce notebook 05's clean mAP (0.387) and 05b's per-image clean AP.

In [ ]:
PRIOR = pd.read_csv(f"{REPO}/prior/05b/per_image_ap.csv")

def compare_with_prior(degradation, severity, restorer, ap_by_img):
    # Share of images whose AP matches notebook 05b exactly (same image, damage seed and restorer).
    p = PRIOR[(PRIOR.degradation == degradation) & (PRIOR.severity == severity) & (PRIOR.restorer == restorer)]
    if p.empty:
        return None
    p = p.set_index("image_id").ap
    diffs = np.array([abs(ap_by_img[i] - p.loc[i]) for i in ap_by_img])
    return float((diffs < 1e-6).mean()), float(diffs.max()), float(np.mean([ap_by_img[i] for i in ap_by_img]) - p.loc[list(ap_by_img)].mean())

t0 = time.time()
clean_results = []
for iid in img_ids:
    clean_results += C.detect(detector, cat_ids, clean_imgs[iid], iid, device)
clean_stats = C.coco_eval(coco_gt, clean_results, img_ids)
clean_map = float(clean_stats[0])
clean_ap = C.per_image_ap(coco_gt, clean_results, img_ids)
match = compare_with_prior("clean", 0, "none", clean_ap)
print(f"clean mAP = {clean_map:.3f} (nb 05: 0.387 on all 300) | per-image AP identical to 05b for {match[0]:.0%} of images "
      f"({time.time() - t0:.0f} s)")
if not QUICK:
    assert abs(clean_map - 0.387) < 0.003, "clean mAP differs from notebook 05: not the same setup"
# Recovery shares and the H3b threshold use the clean mAP of the images actually evaluated: 0.387 in the full run
# (asserted above), the 40-image value in QUICK mode.
CLEAN_MAP_REF = clean_map

## 5 · The sweep

For each condition the damaged images are made once (seeded exactly as notebooks 05/05b), then every restorer is applied
to the identical inputs. For each image: one hooked `predict()` on the clean image and one on the restored image give
the detections and both sets of feature maps; we keep only the distances (17 layers × 4 numbers), not the maps.

New runs, not in 05b:
- **dark_only ℓ = 0.8**: the low-light curve (gain + gamma) with no shot or read noise, to separate darkness from noise;
- **gaussian_seed1 σ = 25**: the same noise level with a different seed, to measure how repeatable per-image AP loss is.

Results are saved after every run, so an interrupted session resumes where it stopped.

In [ ]:
dncnn_net, ck = RS.load_dncnn(CKPT_PATH, device)
print(f"DnCNN: trained {ck['epoch'] + 1} epochs")
RESTORERS = RS.build_restorers(RS.make_dncnn_restore(dncnn_net, device))

LL = ["none", "classical", "dncnn", "clahe_L", "gamma", "gamma>dncnn", "dncnn>gamma"]
CONDITIONS = [                       # (degradation, severity, restorers)
    ("gaussian", 15, ["none", "classical", "dncnn"]),
    ("gaussian", 25, ["none", "classical", "dncnn", "bm3d"]),
    ("gaussian", 50, ["none", "classical", "dncnn", "bm3d"]),
    ("jpeg", 30, ["none", "classical", "dncnn"]),
    ("jpeg", 10, ["none", "classical", "dncnn"]),
    ("low_light", 0.6, LL),
    ("low_light", 0.8, LL),
    ("dark_only", 0.8, ["none", "gamma", "clahe_L", "dncnn"]),
    ("gaussian_seed1", 25, ["none"]),
]
print(sum(len(r) for _, _, r in CONDITIONS) + 1, "runs including clean")

def make_damaged(name, s, iid):
    # Identical to notebooks 05/05b: seed = [SEED, position in the full subset, degradation index, severity index].
    if name == "dark_only":
        return RS.dark_only(clean_imgs[iid], s)
    base, seed = ("gaussian", SEED + 1) if name == "gaussian_seed1" else (name, SEED)
    sev_idx = DEGRADATIONS[base][1].index(s)
    rng = np.random.default_rng([seed, POS[iid], list(DEGRADATIONS).index(base), sev_idx])
    return degrade(clean_imgs[iid], base, s, rng)

In [ ]:
RUNS_CSV, PI_CSV, DIST_CSV = f"{OUT_DIR}/runs.csv", f"{OUT_DIR}/per_image.csv", f"{OUT_DIR}/feature_distances.csv.gz"
runs_rows = pd.read_csv(RUNS_CSV).to_dict("records") if os.path.exists(RUNS_CSV) else []
done = {(r["degradation"], r["severity"], r["restorer"]) for r in runs_rows}
keep = lambda df: df[[(a, b, c) in done for a, b, c in zip(df.degradation, df.severity, df.restorer)]]
pi_rows = keep(pd.read_csv(PI_CSV)).to_dict("records") if os.path.exists(PI_CSV) else []      # drop half-saved runs
dist_rows = keep(pd.read_csv(DIST_CSV, dtype={"layer": str})).to_dict("records") if os.path.exists(DIST_CSV) else []

if ("clean", 0, "none") not in done:
    pi_rows += [dict(degradation="clean", severity=0, restorer="none", image_id=i, ap=clean_ap[i], psnr=np.inf, mse=0.0)
                for i in img_ids]
    runs_rows.append(dict(degradation="clean", severity=0, restorer="none", map=clean_map, map50=float(clean_stats[1]),
                          psnr=np.inf, mse=0.0, seconds=0))

def save():
    pd.DataFrame(dist_rows).to_csv(DIST_CSV, index=False)
    pd.DataFrame(pi_rows).to_csv(PI_CSV, index=False)
    pd.DataFrame(runs_rows).to_csv(RUNS_CSV, index=False)      # last: a run counts as done only once all three exist
save()

t_all = time.time()
for name, s, rests in CONDITIONS:
    todo = [r for r in rests if (name, s, r) not in done]
    if not todo:
        continue
    damaged = [make_damaged(name, s, iid) for iid in img_ids]
    rname_for_tables = "gaussian" if name == "gaussian_seed1" else name     # classical/BM3D settings are per damage type
    for rest in todo:
        t0 = time.time()
        if rest == "bm3d":
            outs = RS.bm3d_many(damaged, s)
        else:
            outs = [np.clip(RESTORERS[rest](img, rname_for_tables, s), 0, 1).astype(np.float32) for img in damaged]
        results, mses = [], []
        for iid, out in zip(img_ids, outs):
            _, f_clean = detect_and_tap(clean_imgs[iid], iid)
            res, f_out = detect_and_tap(out, iid)
            results += res
            for layer, m in F.compare_feature_sets(f_clean, f_out, *out.shape[:2]).items():
                dist_rows.append(dict(degradation=name, severity=s, restorer=rest, image_id=iid, layer=str(layer), **m))
            mses.append(float(np.mean((out - clean_imgs[iid]) ** 2)))
        stats = C.coco_eval(coco_gt, results, img_ids)
        ap = C.per_image_ap(coco_gt, results, img_ids)
        for iid, m in zip(img_ids, mses):
            pi_rows.append(dict(degradation=name, severity=s, restorer=rest, image_id=iid, ap=ap[iid],
                                psnr=10 * np.log10(1 / max(m, 1e-12)), mse=m))
        psnr_mean = float(np.mean([10 * np.log10(1 / max(m, 1e-12)) for m in mses]))
        runs_rows.append(dict(degradation=name, severity=s, restorer=rest, map=float(stats[0]), map50=float(stats[1]),
                              psnr=psnr_mean, mse=float(np.mean(mses)), seconds=round(time.time() - t0)))
        done.add((name, s, rest)); save()
        match = compare_with_prior(name, s, rest, ap)
        msg = "" if match is None else (f" | same per-image AP as 05b: {match[0]:.0%}"
                                        + ("" if match[0] == 1 else f" (max |diff| {match[1]:.3f}, mean diff {match[2]:+.4f})"))
        neck = np.mean([r["cos"] for r in dist_rows[-len(img_ids) * len(tap.layers + ['input']):] if r["layer"] in A.NECK])
        print(f"{name:14s} {s!s:>4} {rest:12s} mAP {stats[0]:.3f}  PSNR {psnr_mean:5.2f}  neck cos {neck:.4f}  "
              f"({time.time() - t0:.0f} s){msg}")
print(f"sweep done in {(time.time() - t_all) / 60:.1f} min")

In [ ]:
dist = pd.read_csv(DIST_CSV, dtype={"layer": str})
pi = pd.read_csv(PI_CSV)
runs = pd.read_csv(RUNS_CSV)
# Sanity: every run has every image at every layer, and nothing is NaN
n_layers = dist.layer.nunique()
counts = dist.groupby(A.KEY).size()
assert (counts == n_layers * len(img_ids)).all(), counts[counts != n_layers * len(img_ids)]
assert dist[["cos", "cka", "rel_l2"]].notna().all().all() and (dist[["cos", "rel_l2"]] >= -1e-9).all().all()
# The prior runs: does mAP match 05b?
prior_runs = pd.read_csv(f"{REPO}/prior/05b/sweep.csv").set_index(A.KEY)["map"]
chk = runs.set_index(A.KEY)["map"].to_frame("map_07").join(prior_runs.rename("map_05b"), how="inner")
chk["diff"] = chk.map_07 - chk.map_05b
if QUICK:
    print("QUICK: mAP here is on 40 images and 05b's on 300, so only the per-image matches above are comparable.")
else:
    print(chk.round(4).to_string())
if not QUICK:
    assert chk["diff"].abs().max() < 0.003, "mAP differs from 05b on a shared run"

## 6 · Depth profiles: where does damage show up? (H1)

Mean distance between damaged and clean features at every layer, with no restoration. Three distances, because they see
different things: **cosine** ignores how loud each activation vector is, **CKA** asks whether the same spatial pattern
is represented at all, **relative L2** sees everything including pure loudness changes. The shaded columns are the
three maps the Detect head reads.

In [ ]:
PROFILE_CONDS = [("gaussian", 15), ("gaussian", 25), ("gaussian", 50), ("jpeg", 30), ("jpeg", 10),
                 ("low_light", 0.6), ("low_light", 0.8), ("dark_only", 0.8)]
fig = P.depth_profiles(dist, PROFILE_CONDS); fig.savefig(f"{OUT_DIR}/fig_depth_profiles.png", dpi=150); plt.show()
lm = A.layer_means(dist, "cos")
print("cosine distance by layer, no restoration:")
print(lm.loc[[k for k in lm.index if k[2] == "none" and k[0] != "clean"]].round(4).to_string())

## 7 · Does DnCNN restore deep features? (H2, H5)

**Feature-damage removal** at a layer = 1 − (distance after restoration) / (distance before). 1 means the restored
image's features are as close to clean as the clean image's own; 0 means no better than the damaged image; below 0
means restoration moved the features *further* from clean. Dotted lines: the same ratio for pixel MSE. Dashed lines:
the share of lost mAP recovered (notebook 05's measure).

In [ ]:
rec = A.recovery_table(dist, runs, CLEAN_MAP_REF, "cos")
cols = ["degradation", "severity", "restorer", "map", "psnr", "R_pixel", "R_input", "R_0", "R_early", "R_neck", "map_recovery"]
print(rec[cols].sort_values(["degradation", "severity", "restorer"]).round(3).to_string(index=False))
rec.round(4).to_csv(f"{OUT_DIR}/recovery_table.csv", index=False)
fig = P.removal_by_layer(dist, runs, CLEAN_MAP_REF, "gaussian", [15, 25, 50], ["classical", "dncnn", "bm3d"])
fig.savefig(f"{OUT_DIR}/fig_removal_gaussian.png", dpi=150); plt.show()
fig = P.removal_by_layer(dist, runs, CLEAN_MAP_REF, "jpeg", [30, 10], ["classical", "dncnn"])
fig.savefig(f"{OUT_DIR}/fig_removal_jpeg.png", dpi=150); plt.show()

## 8 · Low light: why CLAHE fails (H3)

Notebook 05: CLAHE raises PSNR but lowers mAP. If the explanation is "it amplifies the noise", then (a) CLAHE should
push the *early* features further from clean even while it lowers pixel error; (b) darkness without noise should cost
much less; (c) on noise-free dark images, brightening should help. The activation-energy plot (exploratory) shows
whether damaged and brightened images make YOLO's activations quieter or louder than clean.

In [ ]:
fig = P.low_light_panel(dist, runs, (0.6, 0.8)); fig.savefig(f"{OUT_DIR}/fig_low_light.png", dpi=150); plt.show()
fig = P.removal_by_layer(dist, runs, CLEAN_MAP_REF, "low_light", [0.6, 0.8], ["clahe_L", "gamma", "dncnn", "gamma>dncnn"])
fig.savefig(f"{OUT_DIR}/fig_removal_low_light.png", dpi=150); plt.show()
fig = P.norm_profiles(dist, [("low_light", 0.8, r) for r in ("none", "clahe_L", "gamma", "gamma>dncnn")] +
                            [("dark_only", 0.8, r) for r in ("none", "gamma", "clahe_L")])
fig.savefig(f"{OUT_DIR}/fig_activation_energy.png", dpi=150); plt.show()
print(runs[runs.degradation.isin(["low_light", "dark_only"])].round(3).to_string(index=False))

## 9 · Does feature distance predict per-image AP loss? (H4)

Inside one run, every image gets the same damage, so differences between images are about the images. Does the image's
neck distance tell us which images lose the most AP, where PSNR could not (|ρ| ≈ 0.08 in 05b)?

The second-seed run gives a **noise ceiling**: per-image AP loss under two independent draws of the same noise. No
predictor can track AP loss much better than AP loss tracks itself.

Secondary (not pass/fail): the best single layer, chosen on the even-position half of the images and scored on the odd
half, for each of the three distances.

In [ ]:
ceiling = A.noise_ceiling(pi, ("gaussian", 25, "none"), ("gaussian_seed1", 25, "none"))
print(f"noise ceiling (Spearman of per-image AP loss between two noise seeds, sigma 25): {ceiling:.2f}")
t = A.ap_loss_table(dist, pi, "cos", A.NECK)
wr = A.within_run_rhos(t[t.degradation != "clean"])
print("\nwithin-run Spearman with AP loss: neck cosine distance vs minus PSNR")
print(wr.round(2).to_string(index=False))
gr = A.gain_rhos(t)
print("\nrestorer vs none, per image: drop in neck distance vs PSNR gain, as predictors of AP gain")
print(gr.round(2).to_string(index=False))
wr.round(4).to_csv(f"{OUT_DIR}/rho_within_run.csv", index=False); gr.round(4).to_csv(f"{OUT_DIR}/rho_gain.csv", index=False)

fig = P.per_image_rho_by_layer(dist, pi, ceiling); fig.savefig(f"{OUT_DIR}/fig_rho_by_layer.png", dpi=150); plt.show()
fig = P.per_image_scatter(dist, pi, ("gaussian", 50, "none")); fig.savefig(f"{OUT_DIR}/fig_scatter_g50.png", dpi=150); plt.show()
for metric in A.METRICS:
    best, table, neck_odd = A.split_half_layer_choice(dist, pi, metric)
    print(f"{metric:7s} best layer on even half: {best:>5s}  -> odd half rho {table.loc[best, 'rho_odd (test)']:.3f} "
          f"| neck mean on odd half {neck_odd:.3f}")

## 10 · Ranking restorers (H6)

Inside each damaged condition: is the restorer that ends up closest to clean in the neck also the one with the best
mAP? And the highest-PSNR one?

In [ ]:
main = {("gaussian", 15), ("gaussian", 25), ("gaussian", 50), ("jpeg", 30), ("jpeg", 10), ("low_light", 0.6), ("low_light", 0.8)}
ra = A.ranking_agreement(dist, runs, "cos", main)
print(ra.round(2).to_string(index=False))
ra.to_csv(f"{OUT_DIR}/ranking_agreement.csv", index=False)

## 11 · Scorecard, and the layers for Part B

Every verdict below is computed with the rule written in `PLAN.md` before the run. `True` = supported, `False` = not
supported, `None` = the run needed for it is missing (only in `QUICK` mode or after an interruption).

In [ ]:
def to_py(x):
    if isinstance(x, dict):  return {str(k): to_py(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):  return [to_py(v) for v in x]
    if isinstance(x, (np.floating, float)):  return None if not np.isfinite(x) else round(float(x), 4)
    if isinstance(x, (np.integer,)):  return int(x)
    if isinstance(x, (np.bool_,)):  return bool(x)
    return x

sc = A.scorecard(dist, pi, runs, CLEAN_MAP_REF)
for k, (verdict, ev) in sc.items():
    print(f"{k:4s} {str(verdict):5s}  {json.dumps(to_py(ev))[:220]}")
pick, options = A.choose_part_b_layers(dist, pi)
print("\nPart B options (held-out Spearman with AP loss on the Gaussian runs):")
print(options.round(3).to_string(index=False))
print("-> Part B uses:", pick)
json.dump(to_py({"QUICK": QUICK, "n_images": len(img_ids), "clean_map": clean_map, "noise_ceiling": ceiling,
                 "verdicts": {k: v for k, (v, _) in sc.items()}, "evidence": {k: e for k, (_, e) in sc.items()}}),
          open(f"{OUT_DIR}/scorecard_07.json", "w"), indent=1)
json.dump(to_py({"metric": pick["metric"], "layers": list(pick["layers"]), "rho_odd": pick["rho_odd"],
                 "rule": "PLAN.md section 1.4", "options": options.to_dict("records")}),
          open(f"{OUT_DIR}/layers_for_partB.json", "w"), indent=1)
print("outputs:", sorted(os.listdir(OUT_DIR)))

## Findings

*Written after the run, from the scorecard and the figures above.*